# Practice: Pandas Fundamentals (Titanic)

Exercise from **Luis Junco**, Slack `#2--class-activities`, 17 Sept 12:43
(gist: *Exercise to practice Pandas fundamentals*).

- Iteration 1: import the dataset
- Iteration 2: explore the data
- Iteration 3: sort the data
- Iteration 4: check missing values
- Bonus 1: sort by two columns
- Bonus 2: find the column with most missing values and clean it up

---
## Iteration 1: Import the dataset

The gist uses `seaborn`:

```python
import seaborn as sns
titanic = sns.load_dataset('titanic')
```

seaborn is **not installed** on this machine, and `load_dataset` only downloads that same CSV
from GitHub - so we read the file directly with pandas. Identical DataFrame, no install.

In [1]:
import pandas as pd

# already downloaded to 01_python/data/titanic.csv
titanic = pd.read_csv("data/titanic.csv")

titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
## Iteration 2: Explore the data

In [2]:
# first 5 rows
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
# shape: (rows, columns)
print(titanic.shape)

# column names as a list
print(titanic.columns.tolist())

(891, 15)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


---
## Iteration 3: Sort the data

Sort by `fare`, highest first.

In [4]:
titanic.sort_values("fare", ascending=False).head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
679,1,1,male,36.0,0,1,512.3292,C,First,man,True,B,Cherbourg,yes,False
258,1,1,female,35.0,0,0,512.3292,C,First,woman,False,NaN,Cherbourg,yes,True
737,1,1,male,35.0,0,0,512.3292,C,First,man,True,B,Cherbourg,yes,True
88,1,1,female,23.0,3,2,263.0000,S,First,woman,False,C,Southampton,yes,False
438,0,1,male,64.0,1,4,263.0000,S,First,man,True,C,Southampton,no,False


Three passengers paid the same top fare of **512.33** - all in First class.

---
## Iteration 4: Check missing values

`isna()` turns the whole table into True/False (True = missing).

In [5]:
# is there ANY missing value in the whole DataFrame?
print(titanic.isna().values.any())

True


In [6]:
# how many per column?
titanic.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

Three columns have gaps: **deck** (688), **age** (177), **embarked** / **embark_town** (2 each).
Everything else is complete - 891 rows.

---
## Bonus 1: sort by two columns

`age` descending, and where two passengers share an age, `fare` ascending.

In [7]:
titanic.sort_values(["age", "fare"], ascending=[False, True])[["age", "fare", "class"]].head(10)

,age,fare,class
630,80.0,30.0000,First
851,74.0,7.7750,Third
96,71.0,34.6542,First
493,71.0,49.5042,First
116,70.5,7.7500,Third
672,70.0,10.5000,Second
745,70.0,71.0000,First
33,66.0,10.5000,Second
280,65.0,7.7500,Third
456,65.0,26.5500,First


The oldest passenger on board was **80**, travelling First class for a fare of 30.

---
## Bonus 2: drop the worst column, fill the gaps in `age`

In [8]:
missing_counts = titanic.isna().sum()
col_with_most_missing = missing_counts.idxmax()

print("column with most missing values:", col_with_most_missing,
      "(", missing_counts.max(), "of", len(titanic), "rows )")

column with most missing values: deck ( 688 of 891 rows )


In [9]:
titanic_clean = titanic.drop(columns=[col_with_most_missing])
titanic_clean["age"] = titanic_clean["age"].fillna(titanic_clean["age"].mean())

print("missing values left:", titanic_clean.isna().sum().sum())
titanic_clean.isna().sum()

missing values left: 4


survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       2
class          0
who            0
adult_male     0
embark_town    2
alive          0
alone          0
dtype: int64

### Careful - it is not zero yet

The gist's solution says the result should have **0** missing values, but it comes to **4**:
`embarked` and `embark_town` still have 2 each. Dropping `deck` and filling `age` does not
touch them.

To really reach zero, fill those two as well - with the most frequent value (the mode):

In [10]:
titanic_clean["embarked"] = titanic_clean["embarked"].fillna(titanic_clean["embarked"].mode()[0])
titanic_clean["embark_town"] = titanic_clean["embark_town"].fillna(titanic_clean["embark_town"].mode()[0])

print("missing values left:", titanic_clean.isna().sum().sum())
print("shape:", titanic_clean.shape, "(one column fewer than the original", titanic.shape, ")")

missing values left: 0
shape: (891, 14) (one column fewer than the original (891, 15) )


---
## What this exercise teaches

| Task | Method |
|---|---|
| first rows | `df.head()` |
| size | `df.shape` |
| column names | `df.columns.tolist()` |
| sort by one column | `df.sort_values("fare", ascending=False)` |
| sort by two | `df.sort_values(["age", "fare"], ascending=[False, True])` |
| any gaps at all? | `df.isna().values.any()` |
| gaps per column | `df.isna().sum()` |
| worst column | `df.isna().sum().idxmax()` |
| drop a column | `df.drop(columns=[...])` |
| fill gaps | `df["age"].fillna(df["age"].mean())` |

Numbers replace with the **mean**, text with the **mode** - a number has no "most common
word" and text has no average.